<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS14/datasetPBL/Praktikum1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [2]:
# Load dan gabungkan kedua dataset asli
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))

# Ambil 2 kelas utama untuk klasifikasi biner ('Kafe' vs 'Kedai Kopi')
df_bin = df[df['categories'].isin(["['Kafe']", "['Kedai Kopi']"])].copy()
df_bin['label'] = (df_bin['categories'] == "['Kedai Kopi']").astype(int)
# Mapping kelas: {'Kafe': 0, 'Kedai Kopi': 1}
class_indices = {'Kafe': 0, 'Kedai Kopi': 1}

# Ekstraksi fitur teks TF-IDF dan ubah ke bentuk matriks 64x64x3 (12.288 fitur)
tfidf = TfidfVectorizer(max_features=64 * 64 * 3, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df_bin['text'].astype(str)).toarray()

# Pastikan ukuran kolom tepat 12.288 (64x64x3) dengan padding jika fitur unik lebih sedikit
if X_tfidf.shape[1] < 64 * 64 * 3:
    pad_width = (64 * 64 * 3) - X_tfidf.shape[1]
    X_tfidf = np.pad(X_tfidf, ((0, 0), (0, pad_width)), mode='constant')

X = X_tfidf.reshape(-1, 64, 64, 3)
y = df_bin['label'].values

# Langkah 2.1 & 2.2 - Pembagian Data Training dan Data Testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Class indices:", class_indices)
print("Shape training_set:", X_train.shape)
print("Shape test_set:", X_test.shape)

Class indices: {'Kafe': 0, 'Kedai Kopi': 1}
Shape training_set: (1760, 64, 64, 3)
Shape test_set: (440, 64, 64, 3)


In [3]:
# Langkah 3.1 - Inisiasi Model CNN
cnn = tf.keras.models.Sequential()

# Langkah 3.2 - Pembuatan Layer Konvolusi 1
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

# Langkah 3.3 - Pembuatan Layer Pooling 1
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Langkah 3.5 - Flattening
cnn.add(tf.keras.layers.Flatten())

# Langkah 3.6 - Fully Connected Layer 1 (Input)
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

# Langkah 3.7 - Fully Connected Layer 2 (Output)
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# Langkah 3.8 - Compile Model CNN
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

cnn.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 813,217 (3.10 MB)

 Trainable params: 813,217 (3.10 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
cnn.fit(x=X_train, y=y_train, validation_data=(X_test, y_test), batch_size=32, epochs=25)

Epoch 1/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 171ms/step - accuracy: 0.7159 - loss: 0.6099 - val_accuracy: 0.7205 - val_loss: 0.6061
Epoch 2/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 133ms/step - accuracy: 0.7193 - loss: 0.5787 - val_accuracy: 0.7205 - val_loss: 0.5726
Epoch 3/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 158ms/step - accuracy: 0.7335 - loss: 0.5350 - val_accuracy: 0.7318 - val_loss: 0.5700
Epoch 4/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 159ms/step - accuracy: 0.7619 - loss: 0.4952 - val_accuracy: 0.7341 - val_loss: 0.5627
Epoch 5/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 135ms/step - accuracy: 0.7841 - loss: 0.4575 - val_accuracy: 0.7523 - val_loss: 0.5633
Epoch 6/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 158ms/step - accuracy: 0.8250 - loss: 0.4027 - val_accuracy: 0.7591 - val_loss: 0.5667
Epoch 7/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 163ms/step - accuracy: 0.8585 - loss: 0.3414 - val_accuracy: 0.7705 - val_loss: 0.5808
Epoch 8/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 128ms/step - accuracy: 0.9006 - loss: 0.2694 - val_accuracy:

In [5]:
# Contoh 1 data ulasan baru untuk diprediksi
sample_review = ["kopinya enak banget, tempatnya luas buat nugas dan wfc sampai malam"]

# Transformasi ke bentuk tensor (1, 64, 64, 3)
sample_vec = tfidf.transform(sample_review).toarray()
if sample_vec.shape[1] < 64 * 64 * 3:
    sample_vec = np.pad(sample_vec, ((0, 0), (0, (64 * 64 * 3) - sample_vec.shape[1])), mode='constant')
test_data = sample_vec.reshape(1, 64, 64, 3)

# Prediksi menggunakan model CNN
result = cnn.predict(test_data)
print("Indeks Kelas:", class_indices)
print("Nilai Probabilitas:", result[0][0])

if result[0][0] >= 0.5:
    prediction = 'Kedai Kopi'
else:
    prediction = 'Kafe'

print("Hasil Prediksi:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
Indeks Kelas: {'Kafe': 0, 'Kedai Kopi': 1}
Nilai Probabilitas: 0.9934202
Hasil Prediksi: Kedai Kopi
